In [1]:
import pandas as pd
path = "...311\\"

df = pd.read_csv(path + "311_Service_Requests_from_2010_to_Present_20250711.csv", low_memory=False)

df["Created Date"] = pd.to_datetime(df["Created Date"])

In [2]:
df["Incident Address"] = df["Incident Address"] + ", " + df["City"]

In [3]:
df['Incident Date'] = pd.to_datetime(df['Created Date'])
df['Incident Date'] = df['Incident Date'].apply(lambda x: x.date())

df['Incident Time'] = pd.to_datetime(df['Created Date'])
df['Incident Time'] = df['Incident Time'].apply( lambda d : d.time() )


In [6]:
df['Incident Time'] = df['Incident Time'].astype(str)
df["Incident Hour"] = df['Incident Time'].str.split(":").str[0]
df["Incident Hour"] = df["Incident Hour"].astype(int)

In [8]:
y = list(set(df["Unique Key"].tolist() ) )
print (len(df))
print (len(y))


1810183
1810183


In [9]:
df = df[["Created Date","Incident Date","Incident Time","Incident Hour", "Incident Address", "Closed Date", "Agency", "Agency Name", "Complaint Type", "Descriptor","Location Type", "Resolution Description",
        "City", "Borough", "Community Board", "BBL", "Latitude", "Longitude", "Open Data Channel Type"]]

Add Community District Centroid

In [10]:
import geopandas as gpd

pathshape = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\nycd_26a\\"

# Load shapefile
gdf = gpd.read_file(pathshape + "nycd.shp")

# project to a projected CRS
gdf_proj = gdf.to_crs(epsg=2263)

# calculate centroid there
gdf_proj["centroid"] = gdf_proj.geometry.centroid

# convert centroid back to lat/lon
centroids = gdf_proj.set_geometry("centroid").to_crs(epsg=4326)

gdf["lon"] = centroids.geometry.x
gdf["lat"] = centroids.geometry.y

gdf = gdf[["BoroCD", "lon", "lat"]]

gdf["BoroCD"] = gdf["BoroCD"].astype(str)



Create community board number in 311 data

In [11]:
pd.set_option('chained_assignment', None)

import pandas as pd

# Clean
df['Community Board'] = df['Community Board'].str.strip().str.upper()

# Extract board number (zero-padded to 2 digits)
df['cb_num'] = (
    df['Community Board']
    .str.extract(r'(\d{1,2})')[0]
    .str.zfill(2)
)

# Extract borough name
df['borough_name'] = df['Community Board'].str.extract(r'\d{1,2}\s+(.*)')[0]

# Map borough to code
borough_map = {
    'MANHATTAN': '1',
    'BRONX': '2',
    'BROOKLYN': '3',
    'QUEENS': '4',
    'STATEN ISLAND': '5'
}

df['borough_code'] = df['borough_name'].map(borough_map)

# Final 3-digit community district
df['community_district'] = (
    df['borough_code'] + df['cb_num']
).astype('str')

Merge

In [12]:

gdfDic = gdf.set_index('BoroCD')['lat'].to_dict()
df["lat"] = df["community_district"].map(gdfDic)

gdfDic = gdf.set_index('BoroCD')['lon'].to_dict()
df["lon"] = df["community_district"].map(gdfDic)




Add Community District Name

In [13]:
pathfdny = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\"

In [14]:
dp = pd.read_csv(pathfdny + "New_York_City_Population_By_Community_Districts_20260414.csv")
dp["CD Number"] = dp["CD Number"].astype(str)
dp["CD Number"] = dp["CD Number"].str.split(".").str[0]
dp = dp[dp["CD Number"] != 'nan']

dp['Borough2'] = dp['Borough']

dp['Borough2'] = dp['Borough2'].str.replace('Bronx', '2')
dp['Borough2'] = dp['Borough2'].str.replace('Manhattan', '1')
dp['Borough2'] = dp['Borough2'].str.replace('Brooklyn', '3')
dp['Borough2'] = dp['Borough2'].str.replace('Queens', '4')
dp['Borough2'] = dp['Borough2'].str.replace('Staten Island', '5')

dp['boro_cd'] = dp['Borough2'].astype(str) + dp['CD Number'].astype(str).str.zfill(2)

dp['2010 Population'] = dp['2010 Population'].str.replace(',', '')

dp['2010 Population'] = dp['2010 Population'].astype(int)

dpDic = dp.set_index('boro_cd')['CD Name'].to_dict()
df["Community District Name"] = df["community_district"].map(dpDic)


In [15]:
df['lat'].isna().sum()

21800

In [16]:
df.dtypes

Created Date               datetime64[ns]
Incident Date                      object
Incident Time                      object
Incident Hour                       int32
Incident Address                   object
Closed Date                        object
Agency                             object
Agency Name                        object
Complaint Type                     object
Descriptor                         object
Location Type                      object
Resolution Description             object
City                               object
Borough                            object
Community Board                    object
BBL                               float64
Latitude                          float64
Longitude                         float64
Open Data Channel Type             object
cb_num                             object
borough_name                       object
borough_code                       object
community_district                 object
lat                               

In [17]:
df.Borough

0           BROOKLYN
1          MANHATTAN
2          MANHATTAN
3          MANHATTAN
4             QUEENS
             ...    
1810178       QUEENS
1810179        BRONX
1810180        BRONX
1810181     BROOKLYN
1810182        BRONX
Name: Borough, Length: 1810183, dtype: object

Capitalize first letter of words

In [18]:
df['Complaint Type'] = df['Complaint Type'].str.title()
df['Descriptor'] = df['Descriptor'].str.title()
df['Borough'] = df['Borough'].str.title()
df["Incident Address"] = df["Incident Address"].str.title()

df["Complaint Description"] = df['Complaint Type'] + ", " + df['Descriptor']

df["Community District Name"] = df["Community District Name"] + ", " + df["Borough"]

In [19]:
df["Count"] = 1

df.to_csv(path + "311 Service Requests.csv", index=False)

Subset for noise

In [26]:
print (len(df))
df2 = df[df["Complaint Type"].str.contains("Noise") == True].reset_index(drop=True)
print (len(df2))

df2['lat'] = pd.to_numeric(df2['lat'], errors='coerce')
df2['lon'] = pd.to_numeric(df2['lon'], errors='coerce')

print(df2[['lat', 'lon']].dtypes)

df2['Complaint Description'] = df2['Complaint Description'].str.replace('Noise -', '')
df2['Complaint Description'] = df2['Complaint Description'].str.replace('Noise, Noise,', '')
df2['Complaint Description'] = df2['Complaint Description'].str.replace('Noise,', '')
df2['Complaint Description'] = df2['Complaint Description'].str.replace('Noise:', '')
df2['Complaint Description'] = df2['Complaint Description'].map(str.strip)


df2['Descriptor'] = df2['Descriptor'].str.replace('Noise -', '')
df2['Descriptor'] = df2['Descriptor'].str.replace('Noise, Noise,', '')
df2['Descriptor'] = df2['Descriptor'].str.replace('Noise,', '')
df2['Descriptor'] = df2['Descriptor'].str.replace('Noise:', '')
df2['Descriptor'] = df2['Descriptor'].map(str.strip)


df2["Descriptor"] = df2['Descriptor'].str.split("(").str[0].map(str.strip)
df2["Complaint Description"] = df2['Complaint Description'].str.split("(").str[0].map(str.strip)



1810183
404511
lat    float64
lon    float64
dtype: object


Drop outliers for Incident Date and Incident address = '655 East  230 Street, Bronx'

In [27]:
# target address (match exactly how it appears in your data)
target = "655 East  230 Street, Bronx"

# split dataframe
df_target = df2[df2['Incident Address'] == target]
print (len(df_target))
df_other = df2[df2['Incident Address'] != target]
print (len(df_other))

# drop duplicates ONLY for that address based on Incident Date
df_target = df_target.drop_duplicates(subset=['Incident Date'])
print (len(df_target))

# recombine
df2 = pd.concat([df_target, df_other], ignore_index=True).reset_index(drop=True)

46736
357775
42


Drop lat/lon NANs

In [28]:
df2['lat'].isna().sum()

print (len(df2))

df2 = df2.dropna(subset=['lat'])

print (len(df2))

357817
356694


In [29]:
df2.to_csv(path + "311 Service Requests Noise.csv", index=False)


In [140]:
df2

,Created Date,Incident Date,Incident Time,Incident Address,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,...,Open Data Channel Type,cb_num,borough_name,borough_code,community_district,lat,lon,Community District Name,Complaint Description,Count
0,2025-07-07 20:16:19,2025-07-07,20:16:19,"655 East 230 Street, Bronx",07/07/2025 09:39:15 PM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,...,MOBILE,12,BRONX,2,212,40.886299,-73.851495,"Wakefield, Williamsbridge, Bronx","Residential, Loud Music/Party",1
1,2025-07-06 12:54:55,2025-07-06,12:54:55,"655 East 230 Street, Bronx",07/06/2025 01:34:43 PM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,...,MOBILE,12,BRONX,2,212,40.886299,-73.851495,"Wakefield, Williamsbridge, Bronx","Residential, Loud Music/Party",1
2,2025-07-03 11:16:40,2025-07-03,11:16:40,"655 East 230 Street, Bronx",07/03/2025 11:42:53 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,...,MOBILE,12,BRONX,2,212,40.886299,-73.851495,"Wakefield, Williamsbridge, Bronx","Residential, Loud Music/Party",1
3,2025-06-21 19:27:44,2025-06-21,19:27:44,"655 East 230 Street, Bronx",06/21/2025 07:54:54 PM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,...,MOBILE,12,BRONX,2,212,40.886299,-73.851495,"Wakefield, Williamsbridge, Bronx","Residential, Loud Music/Party",1
4,2025-06-17 14:56:45,2025-06-17,14:56:45,"655 East 230 Street, Bronx",06/17/2025 03:15:30 PM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,...,MOBILE,12,BRONX,2,212,40.886299,-73.851495,"Wakefield, Williamsbridge, Bronx","Residential, Loud Music/Party",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357812,2025-01-01 09:31:07,2025-01-01,09:31:07,NaN,01/01/2025 12:00:48 PM,NYPD,New York City Police Department,Noise - Vehicle,Engine Idling,Street/Sidewalk,...,ONLINE,16,BROOKLYN,3,316,40.668156,-73.910943,"Brownsville, Ocean Hill, Brooklyn","Vehicle, Engine Idling",1
357813,2025-01-01 09:30:22,2025-01-01,09:30:22,"90-10 Grand Central Parkway, East Elmhurst",01/01/2025 11:12:54 AM,NYPD,New York City Police Department,Noise - Commercial,Car/Truck Music,Store/Commercial,...,MOBILE,03,QUEENS,4,403,40.758234,-73.876938,"Jackson Heights, North Corona, Queens","Commercial, Car/Truck Music",1
357814,2025-01-01 09:30:02,2025-01-01,09:30:02,"431 West 25 Street, New York",01/01/2025 10:44:56 AM,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,...,MOBILE,04,MANHATTAN,1,104,40.755422,-73.997537,"Chelsea, Clinton, Manhattan","Street/Sidewalk, Loud Music/Party",1
357815,2025-01-01 09:29:51,2025-01-01,09:29:51,"852 West End Avenue, New York",01/01/2025 10:39:44 AM,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,...,ONLINE,07,MANHATTAN,1,107,40.787166,-73.976126,"West Side, Upper West Side, Manhattan","Street/Sidewalk, Loud Music/Party",1


In [10]:
ct = sorted(list(set(df["Complaint Type"].tolist() )))

In [30]:
df["Location Type"].value_counts()[:50]

Location Type
Street/Sidewalk                  574824
RESIDENTIAL BUILDING             355204
Residential Building/House       242334
Street                           161093
Sidewalk                          84001
Store/Commercial                  28439
Park                              16318
Club/Bar/Restaurant               15462
Above Address                     10477
3+ Family Apt. Building           10364
Park/Playground                    9075
Business                           9045
Restaurant/Bar/Deli/Bakery         7356
Subway                             6088
1-2 Family Dwelling                4855
3+ Family Apartment Building       4601
Mixed Use                          3970
Other (Explain Below)              3491
Highway                            3467
Taxi                               3187
Residential Building               2967
Other                              2757
Yard                               2692
Comercial                          2664
Mobile Food Vendor        

In [64]:
df["Complaint Type"].value_counts()[:50]

Complaint Type
Illegal Parking                   298796
Noise - Residential               222579
Heat/Hot Water                    160087
Blocked Driveway                   86316
Noise - Street/Sidewalk            81322
Unsanitary Condition               46986
Water System                       42864
Street Condition                   41786
Abandoned Vehicle                  36124
Noise - Commercial                 33793
Plumbing                           32577
Dirty Condition                    30343
Noise                              28641
Traffic Signal Condition           27616
Paint/Plaster                      25874
Derelict Vehicles                  23386
Missed Collection                  23092
Encampment                         22649
Noise - Vehicle                    21721
Illegal Dumping                    20477
Door/Window                        20139
Rodent                             18100
General Construction/Plumbing      17879
Homeless Person Assistance         16378
W

In [8]:
df.dtypes

Unique Key                          int64
Created Date                       object
Closed Date                        object
Agency                             object
Agency Name                        object
Complaint Type                     object
Descriptor                         object
Location Type                      object
Incident Zip                       object
Incident Address                   object
Street Name                        object
Cross Street 1                     object
Cross Street 2                     object
Intersection Street 1              object
Intersection Street 2              object
Address Type                       object
City                               object
Landmark                           object
Facility Type                      object
Status                             object
Due Date                           object
Resolution Description             object
Resolution Action Updated Date     object
Community Board                   

In [9]:
df["Descriptor"].value_counts()[:50]

Descriptor
Loud Music/Party                                             232925
ENTIRE BUILDING                                              103593
Blocked Hydrant                                               88689
Banging/Pounding                                              69128
Posted Parking Sign Violation                                 67644
No Access                                                     61665
APARTMENT ONLY                                                56494
Blocked Sidewalk                                              44218
Loud Talking                                                  37740
With License Plate                                            36124
Trash                                                         34278
Pothole                                                       25108
Partial Access                                                24651
PESTS                                                         23898
Derelict Vehicles                    